# `ptof_obs_liveness_detection`

## What this notebook does
Detects liveness and data-quality gaps across the pipeline:
1. **Capability silence** (WARN) — a registered capability hasn't produced output in >grace_hours
2. **Shift context missing** (WARN) — blank shift_date/shift_type/batch_nbr in output records
3. **ETL pipeline health** (CRITICAL) — an upstream ETL task failed, meaning the agent is
   running on stale source data even though it's still producing outputs
4. **Write lag anomalies** (WARN, added 2026-09-18) — write_lag_s (ingestion_ts - called_at) for
   a capability/scheduler_run has degraded beyond its own MAD-based historical bound
5. **ETL run slow** (WARN, added 2026-09-18) — an ETL task's duration_seconds has degraded
   beyond its own MAD-based historical bound, without outright failing

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `02_latency_detection` — runs after `01_bronze_projections`,
  before `06_alert`. (Items 4-5 are the first genuinely latency-related logic in this task —
  previously the task name didn't match its contents.)
- **Upstream:** reads `v_llm_bronze`, `v_etl_bronze` (built by `ptof_obs_bronze_projection`),
  `capability_registry` (human-curated by `ptof_obs_setup_seed`), and (added 2026-09-18)
  `write_lag_baseline` / `etl_duration_baseline` (built by `ptof_obs_nightly_baseline`).
- **Downstream:** `ptof_obs_alert.ipynb` reads `etl_pipeline_health` (CRITICAL) and checks
  `capability_silence`, `shift_context_missing`, `write_lag_anomalies`, `etl_run_slow` (all WARN)
  via direct queries.

## Tables/views touched
- **Reads:** `v_llm_bronze`, `v_etl_bronze`, `capability_registry`, `write_lag_baseline`,
  `etl_duration_baseline`
- **Writes:** `capability_silence`, `shift_context_missing`, `etl_pipeline_health`,
  `write_lag_anomalies`, `etl_run_slow`

## Write lag anomalies / ETL run slow (added 2026-09-18, WARN, provisional)
Both are WARN-tier: `ptof_obs_alert.ipynb` computes and prints them to the job log, but they are
**never persisted to `obs_incidents` and never posted to Teams** — same structural guarantee as
the existing `capability_silence`/`shift_context_missing` WARNs. This is deliberate: these are
newly-introduced MAD-based thresholds (see `threshold_basis`, `status='provisional'`) that
haven't been validated against real incident history yet, so they run in observe-only mode
until proven not to be noisy. Each also rolls row-level anomalies up into an hourly bin
requiring >=3 occurrences before it's written at all, so isolated blips never even reach the
findings table.
- `write_lag_anomalies` catches "still writing, but slower than usual" — distinct from
  `capability_silence` ("stopped writing entirely").
- `etl_run_slow` catches "runs completing, but slower than usual" — distinct from
  `etl_pipeline_health` (outright failure) and `etl_pipeline_staleness` (no run at all).
Neither is a substitute for true per-call inference-latency detection (`latency_ms` on
`ptof_primary__ai_llm_audit_log`), which is blocked — that table has 0 rows in prod.

## Dropped detectors (prod migration 2026-09-10)
- `latency_anomalies` / `latency_anomaly_findings` — no `latency_ms` in prod
- `credential_fastfail_daily` — dev-specific model doesn't exist in prod
- `write_lag_daily` — depends on `latency_ms` for ingest-only computation
- `latency_failures` — depends on `success`, `error_class` (not in prod)
- `capability_health` / `capability_error_rate_alert` / `capability_error_rate_findings` — depends on `success`
- `prompt_size_drift` — no `user_prompt_chars` or `capability_latency_baseline` in prod

In [ ]:
%sql
-- capability_silence — WARN-tier liveness check: has each registered capability produced output
-- within its silence_grace_hours window? Joins capability_registry (active, non-null grace) against
-- v_llm_bronze (where output_type is aliased as capability, generated_at as called_at).
-- Prod capabilities: saa-display (2h), situational-awareness (2h), summary (36h).
-- sev2-insights has silence_grace_hours = NULL (irregular cadence) and is excluded by the WHERE.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_silence AS
SELECT
    r.capability, r.expected_min_daily, r.silence_grace_hours, r.owner,
    count(b.id)      AS calls_last_7d,
    max(b.called_at) AS last_call_at,
    round((unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at)))/3600.0, 1)
                     AS hours_since_last_call
FROM mq_gmdf_dev.oil_obs.capability_registry r
LEFT JOIN mq_gmdf_dev.oil_obs.v_llm_bronze b
       ON b.capability = r.capability
      AND b.called_at >= current_timestamp() - INTERVAL 7 DAYS
WHERE r.active = true AND r.silence_grace_hours IS NOT NULL
GROUP BY 1, 2, 3, 4
HAVING max(b.called_at) IS NULL
    OR unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at))
       > r.silence_grace_hours * 3600;

In [ ]:
%sql
-- shift_context_missing — WARN-tier data-quality check: detects output records where shift
-- context fields (shift_date, shift_type, batch_nbr) are blank or null. These fields enable
-- per-shift and per-batch slicing and AI-to-ISH correlation. Reads v_llm_bronze (where
-- output_type is aliased as capability, generated_at as called_at). Scoped to active
-- capabilities via capability_registry inner join.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.shift_context_missing AS
SELECT
    b.capability,
    count(*)                                                 AS total_calls,
    count_if(coalesce(b.shift_type, '') = '')                AS blank_shift_type,
    count_if(coalesce(cast(b.batch_nbr AS STRING), '') = '') AS blank_batch_nbr,
    count_if(b.shift_date IS NULL)                           AS null_shift_date,
    max(b.called_at)                                         AS last_seen,
    current_timestamp()                                      AS detected_at
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.called_at >= current_timestamp() - INTERVAL 7 DAYS
GROUP BY b.capability
HAVING count_if(coalesce(b.shift_type, '') = '') > 0
    OR count_if(coalesce(cast(b.batch_nbr AS STRING), '') = '') > 0
    OR count_if(b.shift_date IS NULL) > 0;

In [ ]:
%sql
-- etl_pipeline_health — CRITICAL detector: captures any ETL task failure in the last 24 hours.
-- The upstream ETL refreshes ~19 source tables every 10-15 min. When a task fails, the SAA
-- agent continues producing outputs using stale data — capability_silence and pipeline_heartbeat
-- won't fire because the agent is still generating, making this the only early warning.
-- finding_signature keyed on (table_or_view, run_id) so each distinct failure is one incident.
-- Reads v_etl_bronze (pass-through view over mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit).
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_pipeline_health AS
SELECT
    run_id,
    run_timestamp,
    table_or_view,
    status,
    error_message,
    duration_seconds,
    sha2(concat_ws('|', table_or_view, run_id), 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.v_etl_bronze
WHERE status = 'failure'
  AND run_timestamp >= current_timestamp() - INTERVAL 24 HOURS;

In [ ]:
%sql
-- write_lag_anomalies (added 2026-09-18, WARN, provisional -- see threshold_basis) -- flags
-- v_llm_bronze rows whose write_lag_s exceeds that (capability, scheduler_run)'s MAD-based
-- upper_bound_s from write_lag_baseline. Distinct from capability_silence (which catches
-- "stopped writing entirely") -- this catches "still writing, but slower than its own history."
-- Row-level anomalies are logged here, but only rolled up into an hourly (capability,
-- window_start) bin with >=3 anomalous rows becomes a row in this table -- a single slow write
-- is noise; a cluster of 3+ in the same hour is a real degradation. WARN-tier: computed and
-- printed to the job log by ptof_obs_alert, never persisted to obs_incidents or posted to Teams.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.write_lag_anomalies AS
WITH flagged AS (
    SELECT
        b.capability, b.scheduler_run, b.write_lag_s, b.called_at,
        window(b.called_at, '60 minutes').start AS window_start,
        base.upper_bound_s
    FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
    JOIN mq_gmdf_dev.oil_obs.write_lag_baseline base
      ON base.capability = b.capability AND base.scheduler_run = b.scheduler_run
    WHERE b.called_at >= current_timestamp() - INTERVAL 24 HOURS
      AND b.write_lag_s > base.upper_bound_s
)
SELECT
    capability, scheduler_run, window_start,
    count(*)                  AS anomalous_count,
    max(write_lag_s)          AS max_write_lag_s,
    max(upper_bound_s)        AS upper_bound_s,
    sha2(concat_ws('|', capability, scheduler_run, cast(window_start AS STRING)), 256)
                               AS finding_signature,
    current_timestamp()       AS detected_at
FROM flagged
GROUP BY capability, scheduler_run, window_start
HAVING count(*) >= 3;

In [ ]:
%sql
-- etl_run_slow (added 2026-09-18, WARN, provisional -- see threshold_basis) -- flags
-- v_etl_bronze runs whose duration_seconds exceeds that (table_or_view, task_name)'s MAD-based
-- upper_bound_s from etl_duration_baseline. Distinct from etl_pipeline_health (which catches
-- outright failures) and etl_pipeline_staleness (which catches "no run completed at all") --
-- this catches "runs are completing, but taking longer than their own history," an early
-- warning that can precede an actual failure or staleness incident.
-- Row-level slow runs are logged, but only rolled up into an hourly (table_or_view, task_name,
-- window_start) bin with >=3 slow runs becomes a row here, same anti-noise rationale as
-- write_lag_anomalies. WARN-tier: log-only, never persisted to obs_incidents or posted to Teams.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_run_slow AS
WITH flagged AS (
    SELECT
        e.table_or_view, e.task_name, e.duration_seconds, e.run_timestamp,
        window(e.run_timestamp, '60 minutes').start AS window_start,
        base.upper_bound_s
    FROM mq_gmdf_dev.oil_obs.v_etl_bronze e
    JOIN mq_gmdf_dev.oil_obs.etl_duration_baseline base
      ON base.table_or_view = e.table_or_view AND base.task_name = e.task_name
    WHERE e.run_timestamp >= current_timestamp() - INTERVAL 24 HOURS
      AND e.status = 'success'
      AND e.duration_seconds > base.upper_bound_s
)
SELECT
    table_or_view, task_name, window_start,
    count(*)                  AS anomalous_count,
    max(duration_seconds)     AS max_duration_s,
    max(upper_bound_s)        AS upper_bound_s,
    sha2(concat_ws('|', table_or_view, task_name, cast(window_start AS STRING)), 256)
                               AS finding_signature,
    current_timestamp()       AS detected_at
FROM flagged
GROUP BY table_or_view, task_name, window_start
HAVING count(*) >= 3;